# 基于 OpenVINO 的端侧现场交付状态自动说明助手

本 Notebook 用于魔搭灵感流复现，面向企业园区、医院科室、工厂仓库等弱网/内网/隐私敏感场景。

核心流程：交付现场照片 -> VLM 图片理解 -> 结构化交付信息 -> 收件人通知 -> TTS 播报文本 -> Markdown 留档。

> 说明：当前仓库先提供可运行的模板链路。补充真实图片后，可在魔搭灵感流环境中启用 OpenVINO VLM 推理。

## 1. 环境安装

官方 baseline 参考：`https://github.com/openvino-dev-samples/modelscope-workshop`。

重点使用：

- Lab 1：Qwen3-VL 多模态视觉语言理解
- Lab 3：Qwen3-TTS 语音合成
- Lab 2：Qwen3-ASR 作为后续二次追问预留

In [ ]:
# 在魔搭灵感流环境中运行时执行
# !pip install -r requirements.txt

## 2. 导入项目模块

如果你从 GitHub clone 仓库，请先切换到项目根目录。

In [ ]:
from pathlib import Path
import json

project_root = Path.cwd()
print("project_root:", project_root)

In [ ]:
from src.delivery_schema import DeliveryContext, VisionObservation
from src.delivery_generator import generate_delivery_result
from src.delivery_prompts import VLM_DELIVERY_PROMPT, build_delivery_prompt

print("模块导入成功")

## 3. 样例数据

真实参赛素材后续放入 `samples/`，并按 `samples/manifest.example.json` 维护测试集。

这里先使用企业园区样例，保证无模型环境也能跑通完整业务闭环。

In [ ]:
context = DeliveryContext(
    scene="enterprise_office",
    recipient="张工",
    item_hint="文件袋 1 个",
    extra_note="收件人暂时不在工位",
)

mock_observation = VisionObservation(
    item="文件袋",
    location="A 座 3 楼前台右侧桌面",
    landmark="旁边有蓝色文件架和访客登记牌",
    status="包装完整，已放置妥当",
    confidence_note="图片中未看到收件人本人，需要由领取人自行确认",
    raw_description="一个文件袋放在前台右侧桌面，附近有蓝色文件架。",
)

print(context)
print(mock_observation)

## 4. VLM 图片理解 Prompt

真实运行时，将交付照片和下面的 prompt 传给 Qwen3-VL OpenVINO 模型。

In [ ]:
print(VLM_DELIVERY_PROMPT)

## 5. 可选：OpenVINO VLM 推理代码

下面代码参考官方 Lab 1。默认不执行，等你在灵感流环境中下载模型并补充图片后再取消注释。

In [ ]:
# from optimum.intel.openvino import OVModelForVisualCausalLM
# from transformers import AutoProcessor
# from qwen_vl_utils import process_vision_info
#
# model_dir = "lab1-multimodal-vlm/Qwen3-VL-4B-Instruct-int4-ov"
# vlm_model = OVModelForVisualCausalLM.from_pretrained(model_dir, device="AUTO")
# processor = AutoProcessor.from_pretrained(
#     model_dir,
#     min_pixels=256 * 28 * 28,
#     max_pixels=1280 * 28 * 28,
# )
#
# def ask_about_image(image_path: str, question: str) -> str:
#     messages = [{
#         "role": "user",
#         "content": [
#             {"type": "image", "image": f"file://{Path(image_path).resolve()}"},
#             {"type": "text", "text": question},
#         ],
#     }]
#     text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     image_inputs, video_inputs = process_vision_info(messages)
#     inputs = processor(
#         text=[text], images=image_inputs, videos=video_inputs,
#         padding=True, return_tensors="pt"
#     )
#     generated_ids = vlm_model.generate(**inputs, max_new_tokens=512)
#     output_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
#     return processor.tokenizer.decode(output_ids[0], skip_special_tokens=True)

## 6. 生成交付说明

将 VLM 观察结果和业务上下文合并，生成收件人通知、TTS 文本和留档记录。

In [ ]:
result = generate_delivery_result(
    context=context,
    observation=mock_observation,
    followup_question="我还是找不到，旁边还有什么标志？",
)

print(json.dumps(result.to_dict(), ensure_ascii=False, indent=2))

## 7. 收件人通知与 TTS 文本

TTS 第一版先输出播报文本。真实音频合成可接入官方 Lab 3 的 Qwen3-TTS OpenVINO baseline。

In [ ]:
print("【收件人通知】")
print(result.recipient_message)
print("
【TTS 播报文本】")
print(result.tts_text)

## 8. 可选：TTS 音频合成接入点

下面是预留位置。运行环境准备好后，将 `result.tts_text` 传入官方 TTS helper 即可生成音频。

In [ ]:
# 伪代码：按官方 lab3-text-to-speech helper 调整
# from qwen_3_tts_helper import OVQwen3TTSModel
# tts_model = OVQwen3TTSModel.from_pretrained(
#     model_dir="lab3-text-to-speech/Qwen3-TTS-0.6B-fp16-ov",
#     device="AUTO",
# )
# audio = tts_model.synthesize(result.tts_text)
# audio.save("outputs/delivery_tts.wav")

## 9. 二次追问演示

ASR 后续可把用户语音追问转成文本。当前先用文本模拟追问，验证对话逻辑。

In [ ]:
print("追问：我还是找不到，旁边还有什么标志？")
print("回答：", result.followup_answer)

## 10. 导出 Markdown 留档

交付记录可用于企业内网、医院科室交接或工厂工单留档。

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
archive_path = output_dir / "delivery_record.md"
archive_path.write_text(result.archive_markdown, encoding="utf-8")
print("已导出：", archive_path)
print(result.archive_markdown)

## 11. 后续补充素材

请按 `samples/manifest.example.json` 补充企业园区、医院科室、工厂仓库三个场景的真实图片和期望输出。

建议每个场景至少 3 组样例，总计 9 组，用于调整 VLM prompt 和文章截图。